# LegaLoom-Env: Full Curriculum GRPO Training (HF Space edition)

Vanilla HF + PEFT + bitsandbytes — no Unsloth (avoids known GRPO+4-bit dtype bug in Unsloth issue #4891).

Trains across **all four difficulty levels** in a balanced curriculum (easy → medium → hard → expert), 20 GRPO steps per phase = **80 total steps**. Evaluates with 10 episodes per task.

**Designed for HF JupyterLab Space on A10G** (~$1/hr, expect ~3 hours = ~$3 total).

**Outputs in repo root after Cell 9:**
- `reward_curves.png`
- `before_after.png`
- `training_log.json`
- `training_scores.json`

Download via JupyterLab file browser → right-click → Download. Then drop into your local repo, commit, push.

In [ ]:
# Cell 1: Install — vanilla HF stack, no Unsloth
!pip install -q 'transformers>=4.46,<4.50' 'trl>=0.12.0,<0.16' 'peft>=0.13' \
    bitsandbytes accelerate openenv-core==0.2.3 fastapi uvicorn pydantic httpx \
    openai pyyaml datasets matplotlib
print('✓ Installed')

In [ ]:
# Cell 2: Path setup — clone repo into writable HF Space location
import sys, os, subprocess

WORK_DIR = '/data/legaloom-env' if os.path.exists('/data') and os.access('/data', os.W_OK) else os.path.expanduser('~/legaloom-env')

if not os.path.exists(WORK_DIR):
    subprocess.run(
        ['git', 'clone', 'https://huggingface.co/spaces/aarav0202/legaloom-env', WORK_DIR],
        check=True,
    )

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print(f'✓ Working dir: {os.getcwd()}')
print(f'✓ Files: {sorted(os.listdir("."))[:8]} ...')

In [ ]:
# Cell 3: Load model with HF + bitsandbytes 4-bit + PEFT LoRA (5-8 min)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f'✓ Model loaded. dtype={next(model.parameters()).dtype}')

In [ ]:
# Cell 4: Baseline measurement — 10 episodes per task (~10-12 min)
from train_grpo import rollout_episode
from server.legaloom_env_environment import LegaloomEnvironment

model.eval()

print('Measuring baseline (untrained) scores, 10 episodes per task...')
baseline_scores = {}
baseline_per_episode = {}
for task_id in ['task_easy', 'task_medium', 'task_hard', 'task_expert']:
    scores = []
    for seed in range(42, 52):
        env = LegaloomEnvironment()
        result = rollout_episode(model, tokenizer, env, task_id, seed=seed, temperature=0.1)
        scores.append(result['final_reward'])
    baseline_scores[task_id] = sum(scores) / len(scores)
    baseline_per_episode[task_id] = scores
    std = (sum((s - baseline_scores[task_id]) ** 2 for s in scores) / len(scores)) ** 0.5
    print(f'  {task_id}: mean={baseline_scores[task_id]:.3f}  std={std:.3f}  n={len(scores)}')

print(f'\nBaseline avg: {sum(baseline_scores.values()) / 4:.3f}')

In [ ]:
# Cell 5: Curriculum training — 4 phases × 20 steps = 80 total (~2-3 hours)
import os
from train_grpo import build_training_dataset, episode_reward_fn
from trl import GRPOConfig, GRPOTrainer

model.train()

schedule = ['task_easy', 'task_medium', 'task_hard', 'task_expert']
log_history = []

for phase_idx, task_id in enumerate(schedule):
    print(f'\n=== Phase {phase_idx+1}/{len(schedule)}: {task_id} (20 steps) ===')
    dataset = build_training_dataset(task_ids=[task_id], examples_per_task=60)
    if dataset is None:
        print(f'  WARNING: empty dataset for {task_id}, skipping')
        continue

    def _make_reward_fn(tid):
        def fn(prompts, completions, **kwargs):
            clean_kwargs = {k: v for k, v in kwargs.items() if k != 'task_id'}
            return episode_reward_fn(prompts, completions, task_id=tid, **clean_kwargs)
        return fn

    cfg = GRPOConfig(
        learning_rate=5e-6,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_generations=4,
        max_prompt_length=1536,
        max_completion_length=768,
        beta=0.01,
        logging_steps=1,
        max_steps=20,
        output_dir=f'./legaloom_grpo_output/phase_{phase_idx}_{task_id}',
        report_to='none',
        bf16=True,
        fp16=False,
        save_strategy='no',
        gradient_checkpointing=True,
    )
    trainer = GRPOTrainer(
        model=model,
        args=cfg,
        train_dataset=dataset,
        reward_funcs=[_make_reward_fn(task_id)],
        processing_class=tokenizer,
    )
    trainer.train()

    for entry in trainer.state.log_history:
        entry['phase'] = phase_idx
        entry['phase_task_id'] = task_id
    log_history.extend(trainer.state.log_history)
    print(f'  ✓ Phase {phase_idx+1} done. {len(trainer.state.log_history)} entries.')

print(f'\n✓ Curriculum complete. {len(log_history)} log entries across {len(schedule)} phases.')

In [ ]:
# Cell 6: Plot reward curves with phase boundaries marked
import matplotlib.pyplot as plt
import json

with open('training_log.json', 'w') as f:
    json.dump(log_history, f, indent=2, default=str)

rewards, phases, phase_tasks = [], [], []
for e in log_history:
    if 'reward' in e:
        rewards.append(e['reward'])
        phases.append(e.get('phase', 0))
        phase_tasks.append(e.get('phase_task_id', '?'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x = list(range(1, len(rewards) + 1))
ax.plot(x, rewards, 'b-', linewidth=1.5, alpha=0.6, label='Episode reward')

phase_colors = ['#e8f4ff', '#e8ffe8', '#fff5e8', '#ffe8e8']
phase_starts = [0]
current_phase = phases[0] if phases else 0
for i, p in enumerate(phases):
    if p != current_phase:
        phase_starts.append(i)
        current_phase = p
phase_starts.append(len(rewards))
for i in range(len(phase_starts) - 1):
    p_idx = phases[phase_starts[i]] if phase_starts[i] < len(phases) else 0
    ax.axvspan(phase_starts[i] + 1, phase_starts[i + 1], color=phase_colors[p_idx % 4], alpha=0.4)
    midpoint = (phase_starts[i] + phase_starts[i + 1]) / 2 + 1
    task_label = phase_tasks[phase_starts[i]].replace('task_', '') if phase_starts[i] < len(phase_tasks) else '?'
    ax.text(midpoint, 0.95, task_label, ha='center', va='top', fontsize=10, fontweight='bold')

if len(rewards) >= 3:
    window = 3
    ma = [sum(rewards[max(0, i - window + 1):i + 1]) / min(i + 1, window) for i in range(len(rewards))]
    ax.plot(x, ma, 'r-', linewidth=2.5, label='3-step moving avg')

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Success threshold')
ax.set_xlabel('Training Step')
ax.set_ylabel('Episode Reward')
ax.set_title('GRPO Curriculum — Reward across all 4 phases')
ax.set_ylim(0, 1)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

ax2 = axes[1]
losses = [e['loss'] for e in log_history if 'loss' in e]
if losses:
    ax2.plot(range(1, len(losses) + 1), losses, 'g-', linewidth=1.5, alpha=0.8)
ax2.set_xlabel('Training Step')
ax2.set_ylabel('Loss')
ax2.set_title('GRPO Curriculum — Loss')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reward_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ reward_curves.png saved')

In [ ]:
# Cell 7: Trained model evaluation — 10 episodes per task
model.eval()

print('Measuring trained model scores, 10 episodes per task...')
trained_scores = {}
trained_per_episode = {}
for task_id in ['task_easy', 'task_medium', 'task_hard', 'task_expert']:
    scores = []
    for seed in range(42, 52):
        env = LegaloomEnvironment()
        result = rollout_episode(model, tokenizer, env, task_id, seed=seed, temperature=0.1)
        scores.append(result['final_reward'])
    trained_scores[task_id] = sum(scores) / len(scores)
    trained_per_episode[task_id] = scores
    std = (sum((s - trained_scores[task_id]) ** 2 for s in scores) / len(scores)) ** 0.5
    delta = trained_scores[task_id] - baseline_scores[task_id]
    rel = (delta / baseline_scores[task_id] * 100) if baseline_scores[task_id] > 0 else 0
    print(f'  {task_id}: before={baseline_scores[task_id]:.3f}  after={trained_scores[task_id]:.3f}  std={std:.3f}  Δ={delta:+.3f}  ({rel:+.0f}%)')

print(f'\nBaseline avg: {sum(baseline_scores.values()) / 4:.3f}')
print(f'Trained avg:  {sum(trained_scores.values()) / 4:.3f}')
print(f'Lift:         {(sum(trained_scores.values()) - sum(baseline_scores.values())) / 4:+.3f}')

In [ ]:
# Cell 8: Before/After bar chart
import matplotlib.pyplot as plt
import json

tasks = ['task_easy', 'task_medium', 'task_hard', 'task_expert']
labels = ['Easy', 'Medium', 'Hard', 'Expert']
before = [baseline_scores[t] for t in tasks]
after = [trained_scores[t] for t in tasks]

fig, ax = plt.subplots(figsize=(11, 6))
x = range(len(tasks))
w = 0.36
ax.bar([i - w / 2 for i in x], before, w, label='Before GRPO (untrained)', color='#e76f51')
ax.bar([i + w / 2 for i in x], after, w, label='After GRPO (4-phase curriculum)', color='#2a9d8f')
for i in x:
    ax.text(i - w / 2, before[i] + 0.015, f'{before[i]:.3f}', ha='center', fontsize=9)
    ax.text(i + w / 2, after[i] + 0.015, f'{after[i]:.3f}', ha='center', fontsize=9)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Success threshold')
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_ylabel('Average Score (10 episodes)')
ax.set_title('LegaLoom-Env: Before vs After Curriculum GRPO Training')
ax.set_ylim(0, 1)
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('before_after.png', dpi=150, bbox_inches='tight')
plt.show()

with open('training_scores.json', 'w') as f:
    json.dump({
        'baseline': baseline_scores,
        'trained': trained_scores,
        'baseline_per_episode': baseline_per_episode,
        'trained_per_episode': trained_per_episode,
        'n_episodes_per_task': 10,
        'training_schedule': ['task_easy', 'task_medium', 'task_hard', 'task_expert'],
        'steps_per_phase': 20,
    }, f, indent=2)
print('✓ before_after.png + training_scores.json saved')

In [ ]:
# Cell 9: Print README replacement table — download files via JupyterLab file browser
import os

print('=' * 60)
print('PASTE THIS INTO README.md (replaces the existing table):')
print('=' * 60)
print()
print('| Task | Baseline | After GRPO | Δ |')
print('|------|---------:|-----------:|------:|')
for task_id, label in [('task_easy', '`task_easy`'),
                       ('task_medium', '`task_medium`'),
                       ('task_hard', '`task_hard`'),
                       ('task_expert', '`task_expert`')]:
    b = baseline_scores[task_id]
    a = trained_scores[task_id]
    delta_pct = ((a - b) / b * 100) if b > 0 else 0
    bold = '**' if a > b else ''
    print(f'| {label} | {b:.3f} | {bold}{a:.3f}{bold} | {delta_pct:+.0f}% |')
avg_b = sum(baseline_scores.values()) / 4
avg_a = sum(trained_scores.values()) / 4
avg_delta = ((avg_a - avg_b) / avg_b * 100) if avg_b > 0 else 0
print(f'| **Average** | **{avg_b:.3f}** | **{avg_a:.3f}** | **{avg_delta:+.0f}%** |')
print()
print('=' * 60)
print(f'Artifacts saved in: {os.getcwd()}')
print('=' * 60)
print('Download these via JupyterLab file browser → right-click → Download:')
for f in ['reward_curves.png', 'before_after.png', 'training_scores.json', 'training_log.json']:
    full_path = os.path.join(os.getcwd(), f)
    exists = '✓' if os.path.exists(full_path) else '✗ MISSING'
    size = f'{os.path.getsize(full_path) / 1024:.1f} KB' if os.path.exists(full_path) else ''
    print(f'  {exists}  {f}  {size}')
print()
print('After downloading: drop into local repo root, commit, push.')
print('Then go to Space Settings → Hardware → CPU basic to stop GPU billing.')